# Insurance 01 — Public references and mixed-file dossiers
## Purpose
Keep the historical prudential experiment unchanged. Public insurer PDFs are **reference material**, never automatically the applicable policy of a synthetic case. The new corpus lives in `data/insurance_v2`; these notebooks live in `notebooks/insurance`.

This notebook verifies file identities, lists every new dossier artifact, reads actual PDFs and tests six explicit failure modes. All case documents are artificial, not insurer-issued documents. No model call or network request is made.

Run the download command in this folder's README once to obtain the public PDFs.


In [1]:
from pathlib import Path
import sys, json, hashlib
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'backend/app').is_dir())
if str(ROOT / 'backend') not in sys.path:
    sys.path.insert(0, str(ROOT / 'backend'))
from IPython.display import display, Markdown, Image
import pandas as pd
BASE = ROOT / 'data/insurance_v2'
print('Isolated insurance workspace:', BASE)



Isolated insurance workspace: C:\Users\choun\Downloads\Prudential_Evidence_Lab_MVP_Source\prudential_evidence_lab\data\insurance_v2


## 1. Public source inventory
A verified SHA-256 proves file identity, not accuracy, authenticity or contractual applicability. Missing downloads remain visible; this notebook does not silently fetch them.


In [2]:
from ingestion.insurance_workspace import prepare
inventory = prepare(ROOT)
display(pd.DataFrame(inventory).drop(columns=['verified_at'], errors='ignore'))
catalog = json.loads((ROOT / 'docs/insurance/public_corpus.json').read_text())
display(pd.DataFrame(catalog['documents'])[['id','issuer','edition','role','cohort','url']])


,id,status,path,bytes,sha256
0,axa_ma_maison_ipid_2025,LOCAL_PDF_VERIFIED,C:/Users/choun/Downloads/Prudential_Evidence_L...,90450,51bed157cb9b95b554c5fe779e111cf5d9129ba0d42116...
1,axa_ma_maison_terms_2025,LOCAL_PDF_VERIFIED,C:/Users/choun/Downloads/Prudential_Evidence_L...,1344674,787461d9eaf4b49434cd6b23c71eb3423d3a64af692292...
2,foyer_mozaik_ipid_2021,LOCAL_PDF_VERIFIED,C:/Users/choun/Downloads/Prudential_Evidence_L...,211516,70e9ede98af41451d5da3dff207e06e4aa35b57648d51d...
3,foyer_mozaik_brochure,LOCAL_PDF_VERIFIED,C:/Users/choun/Downloads/Prudential_Evidence_L...,1747575,35228d551e51a58930a6d86d0daa22106260e8a2d25664...
4,lalux_habitation_ipid_2023,LOCAL_PDF_VERIFIED,C:/Users/choun/Downloads/Prudential_Evidence_L...,709456,86ae907ec51917979ccb3587fa24d7077feb8cf017b8d8...
5,lalux_easyprotect_general_terms_2022,LOCAL_PDF_VERIFIED,C:/Users/choun/Downloads/Prudential_Evidence_L...,832160,7ad5ed2c0ab49379b4c625a4beaa2f563f51f08f2463e0...
6,lalux_property_claim_form,LOCAL_PDF_VERIFIED,C:/Users/choun/Downloads/Prudential_Evidence_L...,980192,e45ddf11087e73b6d0c627453a05919b117597fc2f64e4...


,id,issuer,edition,role,cohort,url
0,axa_ma_maison_ipid_2025,AXA France IARD,2025-10,product_information,axa_france_ma_maison_970464o,https://media.axa.fr/content/dam/axa-fr/image/...
1,axa_ma_maison_terms_2025,AXA France IARD,2025-10,general_conditions,axa_france_ma_maison_970464o,https://media.axa.fr/content/dam/axa-fr/image/...
2,foyer_mozaik_ipid_2021,Foyer Assurances,2021-09,product_information,foyer_mozaik,https://www.foyer.lu/fr/mydoc/12704
3,foyer_mozaik_brochure,Foyer Assurances,unspecified_on_listing,marketing_brochure,foyer_mozaik,https://www.foyer.lu/fr/mydoc/12712
4,lalux_habitation_ipid_2023,LALUX Assurances,2023,product_information,lalux_habitation_comparison,https://www.lalux.lu/fileadmin/mediatheque/doc...
5,lalux_easyprotect_general_terms_2022,LALUX Assurances,2022-04-15,general_terms,lalux_terms_separate_edition,https://www.lalux.lu/fileadmin/mediatheque/ter...
6,lalux_property_claim_form,LALUX Assurances,unverified,blank_claim_form,lalux_forms,https://www.lalux.lu/fileadmin/mediatheque/doc...


## 2. Generate six separate synthetic dossiers
Each dossier contains a schedule, terms, declaration, an invoice where available, and a clearly labelled diagram. The diagram is a generic visual-extraction fixture, not proof of the declared peril. Files link to the portfolio through claim ID and policy ID. The expected result is kept outside the extraction routine.


In [3]:
from ingestion.insurance_dossiers import generate_dossiers, assess_dossier
folder = BASE / 'synthetic/dossiers'
manifests = generate_dossiers(folder)
files = [{'dossier': p.parent.name, 'file': p.name, 'bytes': p.stat().st_size}
         for p in sorted(folder.rglob('*')) if p.is_file()]
display(pd.DataFrame(files).to_string(index=False))


'               dossier                   file  bytes\n       D001_consistent    claim_statement.pdf   1619\n       D001_consistent inspection_diagram.pdf   3177\n       D001_consistent inspection_diagram.png  38638\n       D001_consistent            invoice.pdf   3884\n       D001_consistent          manifest.json   1320\n       D001_consistent    policy_schedule.pdf   1648\n       D001_consistent              terms.pdf   1679\n  D002_missing_invoice    claim_statement.pdf   1620\n  D002_missing_invoice inspection_diagram.pdf   3176\n  D002_missing_invoice inspection_diagram.png  38734\n  D002_missing_invoice          manifest.json   1223\n  D002_missing_invoice    policy_schedule.pdf   1650\n  D002_missing_invoice              terms.pdf   1679\n     D003_wrong_policy    claim_statement.pdf   1622\n     D003_wrong_policy inspection_diagram.pdf   3176\n     D003_wrong_policy inspection_diagram.png  38794\n     D003_wrong_policy            invoice.pdf   3879\n     D003_wrong_policy     

## 3. Read the actual PDF fields, then compare with expectations
The parser recognises a deliberately restricted English field grammar. It is not a general-purpose document understanding system. READY_FOR_REVIEW does not mean covered. Invoice duplication is checked by invoice reference, not by file hash.


In [4]:
results = []
for manifest, directory in zip(manifests, sorted(folder.iterdir()), strict=True):
    actual = assess_dossier(directory)
    results.append({'scenario': manifest['scenario'], 'expected': manifest['expected_issues'],
                    'observed': actual, 'matches': actual == manifest['expected_issues'],
                    'coverage': manifest['coverage_decision']})
display(pd.DataFrame(results))
assert all(row['matches'] for row in results)


,scenario,expected,observed,matches,coverage
0,consistent,[],[],True,NOT_ASSESSED
1,missing_invoice,[missing_invoice],[missing_invoice],True,NOT_ASSESSED
2,wrong_policy,[policy_mismatch],[policy_mismatch],True,NOT_ASSESSED
3,duplicate_invoice,[duplicate_invoice],[duplicate_invoice],True,NOT_ASSESSED
4,amount_conflict,[amount_mismatch],[amount_mismatch],True,NOT_ASSESSED
5,outside_period,[loss_outside_period],[loss_outside_period],True,NOT_ASSESSED


## 4. Reconstruct a financial table
These invoice grids are synthetic and intentionally clean. The comparison uses extracted cells, not a top-k chunk total. Success here does not establish performance on rotated scans or merged real-world tables.


In [5]:
import pymupdf
with pymupdf.open(folder / 'D001_consistent/invoice.pdf') as document:
    tables = document[0].find_tables().tables
    rows = tables[0].extract()
display(pd.DataFrame(rows[1:], columns=rows[0]))
assert sum(int(row[2]) for row in rows[1:]) == 15_025_000
print('Extracted total: 150,250.00 EUR. Synthetic outlier, not a real invoice.')


Consider using the pymupdf_layout package for a greatly improved page layout analysis.


,Description,Quantity,Amount (EUR cents)
0,Repair materials,1,7512500
1,Labour,1,7512500


Extracted total: 150,250.00 EUR. Synthetic outlier, not a real invoice.


## Interpretation and limits
Expected: six matching cases, one three-row invoice table. Failures should stop promotion rather than invent facts. Public references and synthetic case files are not mixed into a common insurance contract. These are local experiments; the web UI still accepts only its existing synthetic text workflow.
